In [14]:
# ── Chargement et merge des 4 fichiers CSV ──────────────────────
# On charge chaque année séparément car le séparateur de 2024 est différent
# 2021, 2022, 2023 → séparateur point-virgule (;)
# 2024 → séparateur virgule (,)
# On ajoute une colonne 'annee' pour pouvoir filtrer par année dans les analyses
# Puis on empile les 4 tables avec pd.concat

import pandas as pd
import numpy as np

df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2021.csv', sep=';')
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2022.csv', sep=';')
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2023.csv', sep=';')
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - usagers 2024.csv', sep=',')

df_2021['annee'] = 2021
df_2022['annee'] = 2022
df_2023['annee'] = 2023
df_2024['annee'] = 2024

usagers_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)
print(f"✅ Merge OK — {usagers_df.shape[0]:,} lignes x {usagers_df.shape[1]} colonnes")
usagers_df.head()

✅ Merge OK — 506,886 lignes x 17 colonnes


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp,annee
0,202400000001,203 988 581,155 781 758,A01,1,1,3,1,2003.0,2,1,-1,-1,-1,-1,-1,2024
1,202400000001,203 988 582,155 781 759,B01,1,1,1,1,1997.0,4,1,-1,-1,-1,-1,-1,2024
2,202400000002,203 988 579,155 781 757,A01,10,3,3,2,1927.0,5,0,-1,-1,3,3,1,2024
3,202400000002,203 988 580,155 781 757,A01,1,1,1,1,1987.0,4,1,0,-1,3,3,1,2024
4,202400000003,203 988 574,155 781 756,A01,2,2,4,2,2007.0,5,8,0,-1,-1,-1,-1,2024


In [15]:
# ── Remplacement des -1 par NaN ─────────────────────────────────
# Dans le fichier BAAC de l'ONISR, la valeur -1 signifie "non renseigné"
# Ce n'est pas une vraie valeur numérique, on la remplace par NaN
cols_sentinel = ['place', 'grav', 'sexe', 'trajet',
                 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']

usagers_df[cols_sentinel] = usagers_df[cols_sentinel].replace(-1, np.nan)
print("✅ Valeurs -1 remplacées par NaN")

✅ Valeurs -1 remplacées par NaN


In [16]:
# ── Correction des années de naissance aberrantes ───────────────
usagers_df.loc[usagers_df['an_nais'] > 2010, 'an_nais'] = np.nan
print(f"✅ an_nais corrigé")
print(f"Min : {usagers_df['an_nais'].min()}  |  Max : {usagers_df['an_nais'].max()}")
print(f"NaN : {usagers_df['an_nais'].isna().sum()}")

✅ an_nais corrigé
Min : 1912.0  |  Max : 2010.0
NaN : 30811


In [17]:
# ── Suppression des colonnes inutiles ───────────────────────────
usagers_df = usagers_df.drop(columns=['place', 'etatp'])
print(f"✅ Colonnes supprimées")
print(f"Colonnes restantes : {list(usagers_df.columns)}")

✅ Colonnes supprimées
Colonnes restantes : ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'annee']


In [18]:
# ── Vérification finale avant export ────────────────────────────
print(f"Shape final : {usagers_df.shape}")

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': usagers_df.isna().sum(),
    'NaN %': (usagers_df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print("\n--- Vérif aucun -1 restant ---")
for col in usagers_df.select_dtypes(include='number').columns:
    n = (usagers_df[col] == -1).sum()
    if n > 0:
        print(f"  ⚠️  {col} : {n} valeurs -1 restantes")
print("✅ Aucun -1 restant")

print("\n--- Doublons ---")
print(f"Doublons : {usagers_df.duplicated(['Num_Acc','id_usager']).sum()}")

Shape final : (506886, 15)

--- NaN par colonne ---
         NaN count  NaN %
grav           419    0.1
sexe         10632    2.1
an_nais      30811    6.1
trajet       11269    2.2
secu1         9865    1.9
secu2       223921   44.2
secu3       489711   96.6
locp        240813   47.5

--- Vérif aucun -1 restant ---
✅ Aucun -1 restant

--- Doublons ---
Doublons : 0


In [19]:
# ── Export du fichier nettoyé ───────────────────────────────────
usagers_df.to_csv('Dataset/usagers_2021_2024_clean.csv', index=False, sep=';')
print(f"✅ Exporté — {len(usagers_df):,} lignes x {usagers_df.shape[1]} colonnes")

✅ Exporté — 506,886 lignes x 15 colonnes
